In [1]:
#am going to train a spam classifier
#first am going to download the dataset from kaggle

In [2]:
#this code install the transformer, datasets, performance metrics, and the hardware optimization
!pip install -q transformers datasets evaluate accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


In [3]:
#download the dataset from kaggle
import kagglehub

# Download latest version
path = kagglehub.dataset_download("purusinghvi/email-spam-classification-dataset")

print("Path to dataset files:", path)

100%|██████████| 43.0M/43.0M [00:03<00:00, 12.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/purusinghvi/email-spam-classification-dataset/versions/1


In [4]:
import os

print(os.listdir(path))

['combined_data.csv']


In [5]:
import pandas as pd
import numpy as np


In [6]:
df = pd.read_csv("/root/.cache/kagglehub/datasets/purusinghvi/email-spam-classification-dataset/versions/1/combined_data.csv", encoding='latin-1')
print("The shape of the spam dataset is: ", df.shape)
df.head()

The shape of the spam dataset is:  (83448, 2)


,label,text
0,1,ounce feather bowl hummingbird opec moment ala...
1,1,wulvob get your medircations online qnb ikud v...
2,0,computer connection from cnn com wednesday es...
3,1,university degree obtain a prosperous future m...
4,0,thanks for all your answers guys i know i shou...


In [7]:
#let's check the number of columns
print(df.columns)
print()

Index(['label', 'text'], dtype='object')



In [8]:
print("\nMissing values:")
print(df.isnull().sum())


Missing values:
label    0
text     0
dtype: int64


In [9]:
#the dataset has 83448 emails, with two columns the label and the text
#and it has no missing values


In [10]:
#let's inspect the labels
df['label'].value_counts()

,count
label,
1,43910
0,39538


In [11]:
#get the percentage
df['label'].value_counts(normalize=True)*100

,proportion
label,
1,52.619595
0,47.380405


# BERT sample Email Spam Classifier

In [12]:
#let's select one spam email and inspect the raw text only

spam_example = df[df['label']==1].iloc[0]

#df['label']==1 creates a condition selecting; all rows where label is 1
#df[df["label"] == 1] gives us only spam emails
#.iloc takes first spam email
spam_example

,0
label,1
text,ounce feather bowl hummingbird opec moment ala...


In [13]:
print("Label:", spam_example["label"])
print("\nEmail:")
print(spam_example["text"])

Label: 1

Email:
ounce feather bowl hummingbird opec moment alabaster valkyrie dyad bread flack desperate iambic hadron heft quell yoghurt bunkmate divert afterimage


In [14]:
spam_emails = df[df["label"] == 1]

for i in range(10):
    print(f"\n--- Spam example {i} ---")
    print(spam_emails.iloc[i]["text"][:500])


--- Spam example 0 ---
ounce feather bowl hummingbird opec moment alabaster valkyrie dyad bread flack desperate iambic hadron heft quell yoghurt bunkmate divert afterimage

--- Spam example 1 ---
wulvob get your medircations online qnb ikud viagra escapenumber escapenumber levitra escapenumber escapenumber cialis escapenumber escapenumber imitrex escapenumber escapenumber flonax escapenumber escapenumber ultram escapenumber escapenumber vioxx escapenumber escapenumber ambien escapenumber escapenumber valium escapenumber escapenumber xannax escapenumber escapenumber soma escapenumber meridia escapenumber escapenumber escapenumber escapenumber cysfrt have you ever stopped to wonder how muc

--- Spam example 2 ---
university degree obtain a prosperous future money earning power and the prestige that comes with having the career position youÂve always dreamed of diploma from prestigious non accredited unversities based on your present knowledge and professional experience if you qualify 

In [15]:
#we use sample two
spam_emails = df[df["label"] == 1]

example = spam_emails.iloc[2]

text = example["text"]
label = example["label"]

print("Label:", label)
print("\nText:")
print(text)

Label: 1

Text:
university degree obtain a prosperous future money earning power and the prestige that comes with having the career position youÂve always dreamed of diploma from prestigious non accredited unversities based on your present knowledge and professional experience if you qualify no required tests classes books or examinations confidentiality assured escapenumber escapenumber escapenumber escapenumber escapenumber hours a day escapenumber days a week including sundays and holidays sun escapenumber jun escapenumber escapenumber escapenumber escapenumber escapenumbershall borrowfor you have 


In [16]:
#next load transformers
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [17]:
tokens = tokenizer.tokenize(text)
print(tokens)

['university', 'degree', 'obtain', 'a', 'prosperous', 'future', 'money', 'earning', 'power', 'and', 'the', 'prestige', 'that', 'comes', 'with', 'having', 'the', 'career', 'position', 'you', '##ave', 'always', 'dreamed', 'of', 'diploma', 'from', 'prestigious', 'non', 'accredited', 'un', '##vers', '##ities', 'based', 'on', 'your', 'present', 'knowledge', 'and', 'professional', 'experience', 'if', 'you', 'qualify', 'no', 'required', 'tests', 'classes', 'books', 'or', 'examinations', 'confidential', '##ity', 'assured', 'escape', '##num', '##ber', 'escape', '##num', '##ber', 'escape', '##num', '##ber', 'escape', '##num', '##ber', 'escape', '##num', '##ber', 'hours', 'a', 'day', 'escape', '##num', '##ber', 'days', 'a', 'week', 'including', 'sundays', 'and', 'holidays', 'sun', 'escape', '##num', '##ber', 'jun', 'escape', '##num', '##ber', 'escape', '##num', '##ber', 'escape', '##num', '##ber', 'escape', '##num', '##ber', 'escape', '##num', '##bers', '##hall', 'borrow', '##for', 'you', 'have']

In [18]:
#print the number of tokens
print("number of tokens: ", len(tokens))

number of tokens:  106


In [19]:
#convert the tokens to tokenIDS

token_ids = tokenizer.convert_tokens_to_ids(tokens)
token_ids

[2118,
 3014,
 6855,
 1037,
 18241,
 2925,
 2769,
 7414,
 2373,
 1998,
 1996,
 14653,
 2008,
 3310,
 2007,
 2383,
 1996,
 2476,
 2597,
 2017,
 10696,
 2467,
 13830,
 1997,
 9827,
 2013,
 8919,
 2512,
 11459,
 4895,
 14028,
 6447,
 2241,
 2006,
 2115,
 2556,
 3716,
 1998,
 2658,
 3325,
 2065,
 2017,
 7515,
 2053,
 3223,
 5852,
 4280,
 2808,
 2030,
 14912,
 18777,
 3012,
 8916,
 4019,
 19172,
 5677,
 4019,
 19172,
 5677,
 4019,
 19172,
 5677,
 4019,
 19172,
 5677,
 4019,
 19172,
 5677,
 2847,
 1037,
 2154,
 4019,
 19172,
 5677,
 2420,
 1037,
 2733,
 2164,
 14803,
 1998,
 11938,
 3103,
 4019,
 19172,
 5677,
 12022,
 4019,
 19172,
 5677,
 4019,
 19172,
 5677,
 4019,
 19172,
 5677,
 4019,
 19172,
 5677,
 4019,
 19172,
 17198,
 9892,
 17781,
 29278,
 2017,
 2031]

In [20]:
#print the first 20
for token, token_id in zip(tokens[:50], token_ids[:50]):
    print(f"{token:15} -> {token_id}")

university      -> 2118
degree          -> 3014
obtain          -> 6855
a               -> 1037
prosperous      -> 18241
future          -> 2925
money           -> 2769
earning         -> 7414
power           -> 2373
and             -> 1998
the             -> 1996
prestige        -> 14653
that            -> 2008
comes           -> 3310
with            -> 2007
having          -> 2383
the             -> 1996
career          -> 2476
position        -> 2597
you             -> 2017
##ave           -> 10696
always          -> 2467
dreamed         -> 13830
of              -> 1997
diploma         -> 9827
from            -> 2013
prestigious     -> 8919
non             -> 2512
accredited      -> 11459
un              -> 4895
##vers          -> 14028
##ities         -> 6447
based           -> 2241
on              -> 2006
your            -> 2115
present         -> 2556
knowledge       -> 3716
and             -> 1998
professional    -> 2658
experience      -> 3325
if              -> 2065
you       

In [21]:
#adding special characters
encoded = tokenizer(
    text,
    add_special_tokens=True,
    truncation=False
)

print("First 10 IDs:")
print(encoded["input_ids"][:10])

print("\nFirst 10 tokens:")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"][:10]))

print("\nTotal tokens:", len(encoded["input_ids"]))

First 10 IDs:
[101, 2118, 3014, 6855, 1037, 18241, 2925, 2769, 7414, 2373]

First 10 tokens:
['[CLS]', 'university', 'degree', 'obtain', 'a', 'prosperous', 'future', 'money', 'earning', 'power']

Total tokens: 108


In [22]:
#check the end to confirm too
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"][-10:]))

['##ber', 'escape', '##num', '##bers', '##hall', 'borrow', '##for', 'you', 'have', '[SEP]']


In [23]:
encoded

{'input_ids': [101, 2118, 3014, 6855, 1037, 18241, 2925, 2769, 7414, 2373, 1998, 1996, 14653, 2008, 3310, 2007, 2383, 1996, 2476, 2597, 2017, 10696, 2467, 13830, 1997, 9827, 2013, 8919, 2512, 11459, 4895, 14028, 6447, 2241, 2006, 2115, 2556, 3716, 1998, 2658, 3325, 2065, 2017, 7515, 2053, 3223, 5852, 4280, 2808, 2030, 14912, 18777, 3012, 8916, 4019, 19172, 5677, 4019, 19172, 5677, 4019, 19172, 5677, 4019, 19172, 5677, 4019, 19172, 5677, 2847, 1037, 2154, 4019, 19172, 5677, 2420, 1037, 2733, 2164, 14803, 1998, 11938, 3103, 4019, 19172, 5677, 12022, 4019, 19172, 5677, 4019, 19172, 5677, 4019, 19172, 5677, 4019, 19172, 5677, 4019, 19172, 17198, 9892, 17781, 29278, 2017, 2031, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [24]:
print(encoded.keys())

KeysView({'input_ids': [101, 2118, 3014, 6855, 1037, 18241, 2925, 2769, 7414, 2373, 1998, 1996, 14653, 2008, 3310, 2007, 2383, 1996, 2476, 2597, 2017, 10696, 2467, 13830, 1997, 9827, 2013, 8919, 2512, 11459, 4895, 14028, 6447, 2241, 2006, 2115, 2556, 3716, 1998, 2658, 3325, 2065, 2017, 7515, 2053, 3223, 5852, 4280, 2808, 2030, 14912, 18777, 3012, 8916, 4019, 19172, 5677, 4019, 19172, 5677, 4019, 19172, 5677, 4019, 19172, 5677, 4019, 19172, 5677, 2847, 1037, 2154, 4019, 19172, 5677, 2420, 1037, 2733, 2164, 14803, 1998, 11938, 3103, 4019, 19172, 5677, 12022, 4019, 19172, 5677, 4019, 19172, 5677, 4019, 19172, 5677, 4019, 19172, 5677, 4019, 19172, 17198, 9892, 17781, 29278, 2017, 2031, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [25]:
print("Input IDs:")
print(encoded["input_ids"][:20])

print("\nAttention mask:")
print(encoded["attention_mask"][:20])

print("\nToken type IDs:")
print(encoded["token_type_ids"][:20])

Input IDs:
[101, 2118, 3014, 6855, 1037, 18241, 2925, 2769, 7414, 2373, 1998, 1996, 14653, 2008, 3310, 2007, 2383, 1996, 2476, 2597]

Attention mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Token type IDs:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [26]:
#token IDS to embeddings
#we want to see how each token is turned into an embeddding
#unversity --> 2118
#2118 is just an index BERT needs a vector

#load the actual pretrained BERT model
from transformers import BertModel

model = BertModel.from_pretrained("bert-base-uncased")
print(model.config.hidden_size)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


768


In [27]:
#inspect the embedding table
word_embedding_table = model.embeddings.word_embeddings

print(word_embedding_table)
print("\nEmbedding table shape:")
print(word_embedding_table.weight.shape)


Embedding(30522, 768, padding_idx=0)

Embedding table shape:
torch.Size([30522, 768])


In [28]:
#now let's look for the learned embedding vector for unversity
university_id = tokenizer.convert_tokens_to_ids("university")
university_embedding = word_embedding_table.weight[university_id]

print("Token:", "university")
print("Token ID:", university_id)
print("Embedding shape:", university_embedding.shape)

print("\nFirst 10 values:")
print(university_embedding[:10])

Token: university
Token ID: 2118
Embedding shape: torch.Size([768])

First 10 values:
tensor([-0.0580,  0.0031, -0.0010, -0.0238, -0.0036, -0.0114, -0.0255,  0.0226,
         0.0296, -0.0122], grad_fn=<SliceBackward0>)


In [29]:
print("Word embedding table:")
print(model.embeddings.word_embeddings.weight.shape)

print("\nPosition embedding table:")
print(model.embeddings.position_embeddings.weight.shape)

print("\nToken-type embedding table:")
print(model.embeddings.token_type_embeddings.weight.shape)

Word embedding table:
torch.Size([30522, 768])

Position embedding table:
torch.Size([512, 768])

Token-type embedding table:
torch.Size([2, 768])


In [30]:
#test for unversity
# University information
token_id = 2118
position_id = 1
token_type_id = 0

# Retrieve the three vectors
word_emb = model.embeddings.word_embeddings.weight[token_id]
position_emb = model.embeddings.position_embeddings.weight[position_id]
type_emb = model.embeddings.token_type_embeddings.weight[token_type_id]

print("Word embedding:", word_emb.shape)
print(word_emb[:5])

print("\nPosition embedding:", position_emb.shape)
print(position_emb[:5])

print("\nToken-type embedding:", type_emb.shape)
print(type_emb[:5])

Word embedding: torch.Size([768])
tensor([-0.0580,  0.0031, -0.0010, -0.0238, -0.0036], grad_fn=<SliceBackward0>)

Position embedding: torch.Size([768])
tensor([ 0.0078,  0.0023, -0.0194, -0.0171, -0.0132], grad_fn=<SliceBackward0>)

Token-type embedding: torch.Size([768])
tensor([0.0004, 0.0110, 0.0037, 0.0015, 0.0006], grad_fn=<SliceBackward0>)


In [31]:
#cobine the word embedding, position embedding and type embedding
combined = word_emb + position_emb + type_emb

print("Combined shape:", combined.shape)
print("\nFirst 10 values:")
print(combined[:10])

#after this BERT does layerNorm and dropout

Combined shape: torch.Size([768])

First 10 values:
tensor([-0.0498,  0.0163, -0.0168, -0.0394, -0.0162, -0.0082, -0.0206,  0.0079,
         0.0283, -0.0320], grad_fn=<SliceBackward0>)


In [32]:
encoded_tensor = tokenizer(text,
                           add_special_tokens=True,
                           truncation=False,
                           return_tensors="pt")

In [33]:
encoded_tensor

{'input_ids': tensor([[  101,  2118,  3014,  6855,  1037, 18241,  2925,  2769,  7414,  2373,
          1998,  1996, 14653,  2008,  3310,  2007,  2383,  1996,  2476,  2597,
          2017, 10696,  2467, 13830,  1997,  9827,  2013,  8919,  2512, 11459,
          4895, 14028,  6447,  2241,  2006,  2115,  2556,  3716,  1998,  2658,
          3325,  2065,  2017,  7515,  2053,  3223,  5852,  4280,  2808,  2030,
         14912, 18777,  3012,  8916,  4019, 19172,  5677,  4019, 19172,  5677,
          4019, 19172,  5677,  4019, 19172,  5677,  4019, 19172,  5677,  2847,
          1037,  2154,  4019, 19172,  5677,  2420,  1037,  2733,  2164, 14803,
          1998, 11938,  3103,  4019, 19172,  5677, 12022,  4019, 19172,  5677,
          4019, 19172,  5677,  4019, 19172,  5677,  4019, 19172,  5677,  4019,
         19172, 17198,  9892, 17781, 29278,  2017,  2031,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0,

In [34]:
print("Input IDs shape:", encoded_tensor["input_ids"].shape)
print("Attention mask shape:", encoded_tensor["attention_mask"].shape)
print("Token type IDs shape:", encoded_tensor["token_type_ids"].shape)

Input IDs shape: torch.Size([1, 108])
Attention mask shape: torch.Size([1, 108])
Token type IDs shape: torch.Size([1, 108])


In [35]:
#let BERT's embedding module process the IDS
import torch
model.eval()

with torch.no_grad():
  embeddings = model.embeddings(
      input_ids = encoded_tensor["input_ids"],
      token_type_ids = encoded_tensor["token_type_ids"]
  )

print("Embeddings shape:", embeddings.shape)

Embeddings shape: torch.Size([1, 108, 768])


In [36]:
#Rather than jumping straight into the full Transformer,
#let's inspect BERT's first attention layer:

layer_1 = model.encoder.layer[0]
layer_1

BertLayer(
  (attention): BertAttention(
    (self): BertSelfAttention(
      (query): Linear(in_features=768, out_features=768, bias=True)
      (key): Linear(in_features=768, out_features=768, bias=True)
      (value): Linear(in_features=768, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (output): BertSelfOutput(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (intermediate): BertIntermediate(
    (dense): Linear(in_features=768, out_features=3072, bias=True)
    (intermediate_act_fn): GELUActivation()
  )
  (output): BertOutput(
    (dense): Linear(in_features=3072, out_features=768, bias=True)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

In [37]:
attention = layer_1.attention.self
print("Query:")
print(attention.query)

print("\nKey:")
print(attention.key)

print("\nValue:")
print(attention.value)

Query:
Linear(in_features=768, out_features=768, bias=True)

Key:
Linear(in_features=768, out_features=768, bias=True)

Value:
Linear(in_features=768, out_features=768, bias=True)


In [38]:
#let's calculate the Q,K, V
embeddings.shape

torch.Size([1, 108, 768])

In [39]:
with torch.no_grad():
  Q = attention.query(embeddings)
  K = attention.key(embeddings)
  V = attention.value(embeddings)

print("Input X:", embeddings.shape)
print("Q:", Q.shape)
print("K:", K.shape)
print("V:", V.shape)

Input X: torch.Size([1, 108, 768])
Q: torch.Size([1, 108, 768])
K: torch.Size([1, 108, 768])
V: torch.Size([1, 108, 768])


In [40]:
Q

tensor([[[ 0.7090, -0.1532, -0.0324,  ..., -3.0180,  0.9906, -1.5274],
         [-0.5682,  0.3943, -0.3684,  ..., -0.1471,  1.5979, -1.9570],
         [-0.2258,  0.2764,  0.2631,  ..., -1.8836, -0.3695, -1.4900],
         ...,
         [ 0.2766,  0.2600, -0.5971,  ...,  1.0700,  0.1845, -1.4686],
         [-0.1673,  0.4006, -1.5762,  ...,  2.0600,  1.1486, -1.3594],
         [ 0.4537, -0.4572, -0.6036,  ..., -0.0697,  1.9614, -0.6486]]])

In [41]:
#let's get the corresponding q, k and v for unversity
q_university = Q[0, 1]
k_university = K[0, 1]
v_university = V[0, 1]

print("University Q shape:", q_university.shape)
print("University K shape:", k_university.shape)
print("University V shape:", v_university.shape)

print("\nFirst 5 values of University's Q:")
print(q_university[:5])

print("\nFirst 5 values of University's K:")
print(k_university[:5])

print("\nFirst 5 values of University's V:")
print(v_university[:5])

print("\nK and V identical?", torch.allclose(k_university, v_university))

University Q shape: torch.Size([768])
University K shape: torch.Size([768])
University V shape: torch.Size([768])

First 5 values of University's Q:
tensor([-0.5682,  0.3943, -0.3684,  0.7784, -2.3545])

First 5 values of University's K:
tensor([-0.4730, -0.0370, -0.5226, -0.0051, -1.3048])

First 5 values of University's V:
tensor([-0.2305,  0.2079,  0.4035,  1.0769, -0.1095])

K and V identical? False


In [42]:
#Next step: 768 → 12 heads × 64head dimensions
batch_size = Q.shape[0]       # 1
seq_length = Q.shape[1]       # 108

num_heads = 12
head_dim = 64

# First split 768 into 12 × 64
Q_heads = Q.view(batch_size, seq_length, num_heads, head_dim)
K_heads = K.view(batch_size, seq_length, num_heads, head_dim)
V_heads = V.view(batch_size, seq_length, num_heads, head_dim)

print("After splitting:")
print("Q:", Q_heads.shape)

After splitting:
Q: torch.Size([1, 108, 12, 64])


In [43]:
#But for calculating attention, it's more convenient to put the heads before the tokens. So:
Q_heads = Q_heads.permute(0, 2, 1, 3)
K_heads = K_heads.permute(0, 2, 1, 3)
V_heads = V_heads.permute(0, 2, 1, 3)

print("Q heads:", Q_heads.shape)
print("K heads:", K_heads.shape)
print("V heads:", V_heads.shape)

Q heads: torch.Size([1, 12, 108, 64])
K heads: torch.Size([1, 12, 108, 64])
V heads: torch.Size([1, 12, 108, 64])


In [44]:
#[1, 12, 108, 64]
#[email, head, token, dimensions]


In [45]:
#we can finally ask
#Give me the query of the token "university" in attention head 1.
q_university_head1 = Q_heads[0, 0, 1, :]

print("University Q — Head 1:")
print(q_university_head1)

print("\nShape:")
print(q_university_head1.shape)

University Q — Head 1:
tensor([-0.5682,  0.3943, -0.3684,  0.7784, -2.3545, -0.0788, -0.5002, -0.2084,
         0.2782, -0.4221, -0.1167,  0.2338, -0.1410, -0.0191,  0.1005, -0.1762,
        -0.3250, -0.8159, -0.6281, -1.1768, -0.1558,  0.1165,  0.1939,  0.3685,
        -0.0852,  0.0544, -0.5851,  0.4140,  0.0887, -1.8572,  1.0307, -0.5559,
         0.1369,  0.5670, -0.2931, -0.9482,  0.1902, -0.1184, -0.3604, -0.5488,
        -0.4614,  0.2205,  0.0221, -2.7096, -0.6735, -1.6549,  0.2219, -0.0935,
        -0.7615,  0.6584,  0.1165,  0.6005, -0.4354, -0.8996,  0.1303,  0.0099,
         0.2175, -0.1579, -0.5986, -0.5864,  0.4719,  0.7251, -1.2256,  0.1051])

Shape:
torch.Size([64])


In [46]:
#Next: compare University's Q with every token's K
#But Head 1 also contains a key vector for every one of our 108 tokens:
K_head1 = K_heads[0, 0]

print("All keys in Head 1:", K_head1.shape)

All keys in Head 1: torch.Size([108, 64])


In [47]:
#now attention asks for university which of these 108 tokens are relevant
#it compares University's Q agaisnt every K
#here we are doing only for one head, head1
import math

scores = torch.matmul(
    q_university_head1,
    K_head1.T
)

print("Scores shape:", scores.shape)
print("First 10 raw scores:")
print(scores[:10])

Scores shape: torch.Size([108])
First 10 raw scores:
tensor([11.0467, 14.4865, 12.8640, 14.3380,  2.1449, 16.9342,  8.8854, 14.4161,
        18.5741, 10.6036])


In [48]:
#But there's one more step before softmax. Transformers use scaled dot-product attention, so we divide by:
scaled_scores = scores / math.sqrt(64)

print("First 10 scaled scores:")
print(scaled_scores[:10])

First 10 scaled scores:
tensor([1.3808, 1.8108, 1.6080, 1.7922, 0.2681, 2.1168, 1.1107, 1.8020, 2.3218,
        1.3255])


In [49]:
#Next we turn those 108 scaled scores into attention weights using softmax.
import torch.nn.functional as F

attention_weights = F.softmax(scaled_scores, dim=-1)

print("Attention weights shape:", attention_weights.shape)
print("First 10 attention weights:")
print(attention_weights[:10])

print("\nSum of all attention weights:")
print(attention_weights.sum())

Attention weights shape: torch.Size([108])
First 10 attention weights:
tensor([0.0086, 0.0132, 0.0108, 0.0130, 0.0028, 0.0179, 0.0066, 0.0131, 0.0220,
        0.0081])

Sum of all attention weights:
tensor(1.)


In [50]:
#Now let's see which tokens "university" actually gives the highest attention to in Head 1
tokens_with_special = tokenizer.convert_ids_to_tokens(
    encoded_tensor["input_ids"][0]
)

top_values, top_indices = torch.topk(attention_weights, k=10)

for weight, idx in zip(top_values, top_indices):
    print(f"{tokens_with_special[idx]}: {weight.item():.4f}")

accredited: 0.0399
confidential: 0.0226
you: 0.0225
earning: 0.0220
prestigious: 0.0220
career: 0.0204
qualify: 0.0197
##ities: 0.0193
dreamed: 0.0190
prosperous: 0.0179


In [51]:
#Now V finally enters
V_head1 = V_heads[0, 0]   # all 108 value vectors in Head 1

print("V Head 1 shape:", V_head1.shape)

university_context_head1 = torch.matmul(
    attention_weights,
    V_head1
)

print("University context vector — Head 1 shape:",
      university_context_head1.shape)

print("\nFirst 10 values:")
print(university_context_head1[:10])

V Head 1 shape: torch.Size([108, 64])
University context vector — Head 1 shape: torch.Size([64])

First 10 values:
tensor([-0.3630,  0.0178,  0.2574, -0.0337, -0.0465,  0.1095, -0.2833,  0.1371,
        -0.0686,  0.0878])


In [52]:
#The exact same process happens independently in all 12 heads:
#so let's do all tokens accross all 12heads
# Q_heads, K_heads, V_heads:
# [1, 12, 108, 64]

scores_all = torch.matmul(
    Q_heads,
    K_heads.transpose(-2, -1)
)

print("Scores:", scores_all.shape)

scores_all = scores_all / (64 ** 0.5)

weights_all = torch.softmax(scores_all, dim=-1)

print("Attention weights:", weights_all.shape)

context_all = torch.matmul(
    weights_all,
    V_heads
)

print("Context:", context_all.shape)

Scores: torch.Size([1, 12, 108, 108])
Attention weights: torch.Size([1, 12, 108, 108])
Context: torch.Size([1, 12, 108, 64])


In [53]:
#1 email * 12 heads * 108query tokens * 108 key tokens
#In other words, for every head, every token gets to compare its Q against every token's K.

In [54]:
#put the heads back together
# Move token dimension before head dimension
context_transposed = context_all.permute(0, 2, 1, 3)

print("Before concatenation:", context_transposed.shape)

# Concatenate the 12 heads
context_concat = context_transposed.contiguous().view(
    1, 108, 768
)

print("After concatenation:", context_concat.shape)

Before concatenation: torch.Size([1, 108, 12, 64])
After concatenation: torch.Size([1, 108, 768])


In [55]:
university_context = context_concat[0, 1]

print("University after 12 heads:", university_context.shape)
print("First 10 values:", university_context[:10])

University after 12 heads: torch.Size([768])
First 10 values: tensor([-0.3630,  0.0178,  0.2574, -0.0337, -0.0465,  0.1095, -0.2833,  0.1371,
        -0.0686,  0.0878])


In [56]:
print(torch.allclose(
    university_context[:64],
    university_context_head1,
    atol=1e-5
))

True


In [57]:
attention_output_dense = layer_1.attention.output.dense

with torch.no_grad():
    projected_context = attention_output_dense(context_concat)

print("Before output projection:", context_concat.shape)
print("After output projection:", projected_context.shape)

print("\nUniversity before projection:")
print(context_concat[0, 1, :10])

print("\nUniversity after projection:")
print(projected_context[0, 1, :10])

Before output projection: torch.Size([1, 108, 768])
After output projection: torch.Size([1, 108, 768])

University before projection:
tensor([-0.3630,  0.0178,  0.2574, -0.0337, -0.0465,  0.1095, -0.2833,  0.1371,
        -0.0686,  0.0878])

University after projection:
tensor([-0.1214,  0.0695,  0.0386, -0.4253,  0.0255, -0.0683,  0.1600, -0.1788,
         0.0560, -0.5137])


In [58]:
# Original representation entering Layer 1
original_X = embeddings

# Residual connection
residual_sum = original_X + projected_context

# Layer normalization
layer_norm = layer_1.attention.output.LayerNorm

with torch.no_grad():
    attention_block_output = layer_norm(residual_sum)

print("Original X:", original_X.shape)
print("Projected attention:", projected_context.shape)
print("After residual addition:", residual_sum.shape)
print("After LayerNorm:", attention_block_output.shape)

print("\nUniversity original X:")
print(original_X[0, 1, :10])

print("\nUniversity after attention + residual + LayerNorm:")
print(attention_block_output[0, 1, :10])

Original X: torch.Size([1, 108, 768])
Projected attention: torch.Size([1, 108, 768])
After residual addition: torch.Size([1, 108, 768])
After LayerNorm: torch.Size([1, 108, 768])

University original X:
tensor([-0.6342,  0.5094, -0.0019, -0.3071, -0.0812,  0.1054, -0.0818,  0.3874,
         0.7763, -0.2230])

University after attention + residual + LayerNorm:
tensor([-0.8991,  0.8418, -0.2192, -1.4868,  0.2858, -0.4047,  0.6394,  0.5164,
         1.2498, -1.0845])


In [59]:
#Next: Feed-Forward Network
#The feed-forward network is different. It processes the resulting representation of each token independently.
with torch.no_grad():
    intermediate_linear = layer_1.intermediate.dense(
        attention_block_output
    )

    intermediate_output = layer_1.intermediate.intermediate_act_fn(
        intermediate_linear
    )

print("Input to feed-forward:", attention_block_output.shape)
print("After 768 → 3072:", intermediate_linear.shape)
print("After GELU:", intermediate_output.shape)

print("\nUniversity before FFN:")
print(attention_block_output[0, 1, :10])

print("\nUniversity after expansion + GELU:")
print(intermediate_output[0, 1, :10])

Input to feed-forward: torch.Size([1, 108, 768])
After 768 → 3072: torch.Size([1, 108, 3072])
After GELU: torch.Size([1, 108, 3072])

University before FFN:
tensor([-0.8991,  0.8418, -0.2192, -1.4868,  0.2858, -0.4047,  0.6394,  0.5164,
         1.2498, -1.0845])

University after expansion + GELU:
tensor([-0.0755, -0.0010, -0.1669, -0.0022, -0.0126, -0.0297, -0.0039, -0.1286,
        -0.0070, -0.0048])


In [60]:
# 3072 → 768
with torch.no_grad():
    ffn_output = layer_1.output.dense(intermediate_output)

print("After 3072 → 768:", ffn_output.shape)

# Residual connection:
# add the representation that ENTERED the FFN
ffn_residual = attention_block_output + ffn_output

# LayerNorm
with torch.no_grad():
    layer1_output = layer_1.output.LayerNorm(ffn_residual)

print("After residual:", ffn_residual.shape)
print("Final Layer 1 output:", layer1_output.shape)

print("\nUniversity entering FFN:")
print(attention_block_output[0, 1, :10])

print("\nUniversity after complete Layer 1:")
print(layer1_output[0, 1, :10])

After 3072 → 768: torch.Size([1, 108, 768])
After residual: torch.Size([1, 108, 768])
Final Layer 1 output: torch.Size([1, 108, 768])

University entering FFN:
tensor([-0.8991,  0.8418, -0.2192, -1.4868,  0.2858, -0.4047,  0.6394,  0.5164,
         1.2498, -1.0845])

University after complete Layer 1:
tensor([-0.9270,  0.7738, -0.1093, -0.8566,  0.1085,  0.0422, -0.2872,  0.6377,
         0.9146, -0.5477])


we've manually followed "university" from its original token ID 2118, through embeddings, Q/K/V, 12-head self-attention, and the feed-forward network—all the way through one complete BERT Transformer layer. Then we can see what happens with Layers 2–12.

In [61]:
#And Layer 2 has its own learned \(W_Q,W_K,W_V,W_O\) and FFN parameters. It does not reuse Layer 1's matrices.

Next step
We don't need to manually repeat all those calculations 11 more times. We understand the mechanism now.
Let's let BERT run the entire 12-layer encoder and inspect "university" after the final layer:

In [62]:
with torch.no_grad():
    outputs = model(
        input_ids=encoded_tensor["input_ids"],
        attention_mask=encoded_tensor["attention_mask"],
        token_type_ids=encoded_tensor["token_type_ids"]
    )

final_hidden_states = outputs.last_hidden_state

print("Final BERT output:", final_hidden_states.shape)

final_university = final_hidden_states[0, 1]

print("\nFinal university representation:", final_university.shape)
print("First 10 values:")
print(final_university[:10])

Final BERT output: torch.Size([1, 108, 768])

Final university representation: torch.Size([768])
First 10 values:
tensor([ 0.0431,  0.0571, -0.0189, -0.2936,  0.5953,  0.1827, -0.1527,  0.5225,
         0.1861, -0.2802])


We have 108 final token vectors. Which one do we use to classify the entire email as spam/not spam?

Remember the CLS
For sequence classification, we use the final representation associated with [CLS] as the sequence-level representation

In [63]:
#LET'S INSPECT IT
cls_representation = final_hidden_states[0, 0]

print("Final [CLS] representation shape:")
print(cls_representation.shape)

print("\nFirst 10 values:")
print(cls_representation[:10])

Final [CLS] representation shape:
torch.Size([768])

First 10 values:
tensor([ 0.0657,  0.1553,  0.0606,  0.1183, -0.2627, -0.7283,  0.3646,  0.2373,
        -0.3333, -0.4027])


In [64]:
import torch.nn as nn

classifier = nn.Linear(768, 2)

print(classifier)
print("Weight shape:", classifier.weight.shape)
print("Bias shape:", classifier.bias.shape)

Linear(in_features=768, out_features=2, bias=True)
Weight shape: torch.Size([2, 768])
Bias shape: torch.Size([2])


In [65]:
# One very important thing: don't interpret this classifier's predictions yet.
# We just created nn.Linear(768, 2), so its weights are random. It has not learned spam classification.

#NOW WE HAVE OUR CLASSIFICATION HEAD

with torch.no_grad():
    logits = classifier(cls_representation)

print("Logits:", logits)
print("Logits shape:", logits.shape)

Logits: tensor([0.0404, 0.2766])
Logits shape: torch.Size([2])


In [66]:
#THOSE TWO NUMBERS ABOVE ARE SCORES NOT PROBABILITIES
probabilities = torch.softmax(logits, dim=-1)

print("Probabilities:", probabilities)
print("Sum:", probabilities.sum())

Probabilities: tensor([0.4412, 0.5588])
Sum: tensor(1.)


In [67]:
#we get the loss
import torch.nn.functional as F

# Cross entropy expects:
# logits shape: [batch_size, number_of_classes]
# labels shape: [batch_size]

logits_batch = logits.unsqueeze(0)   # [2] -> [1, 2]
true_label = torch.tensor([1])       # Spam

loss = F.cross_entropy(logits_batch, true_label)

print("Logits:", logits_batch)
print("True label:", true_label)
print("Loss:", loss.item())

Logits: tensor([[0.0404, 0.2766]])
True label: tensor([1])
Loss: 0.5820240378379822


# BERT Email Spam Classifier — Fine-Tuning

In [68]:
#we start by splitting the dataset

from sklearn.model_selection import train_test_split

# First: 80% train, 20% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

# Split the remaining 20% equally:
#we take in the temp_df which has the 20% and split it
# 10% validation, 10% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))
print("Test examples:", len(test_df))

print("\nTraining label distribution:")
print(train_df["label"].value_counts(normalize=True))

print("\nValidation label distribution:")
print(val_df["label"].value_counts(normalize=True))

print("\nTest label distribution:")
print(test_df["label"].value_counts(normalize=True))

Training examples: 66758
Validation examples: 8345
Test examples: 8345

Training label distribution:
label
1    0.526199
0    0.473801
Name: proportion, dtype: float64

Validation label distribution:
label
1    0.526183
0    0.473817
Name: proportion, dtype: float64

Test label distribution:
label
1    0.526183
0    0.473817
Name: proportion, dtype: float64


In [69]:
#The stratification worked exactly as intended. All three sets preserve essentially the same class balance: about 52.6% spam and 47.4% not spam.


In [71]:
#tokenization
#previously we did this text --> wordpiece tokens --> inputids
#but bert needs attention mask
#because emails have different lengths
#one might produce 108 tokens, another 40 tokens
#but bert can only accept 512 tokens and examples
#in the same batch need compatible lengths
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

MAX_LENGTH = 256

print("Vocabulary size:", tokenizer.vocab_size)
print("Maximum model length:", tokenizer.model_max_length)
print("Training max length we chose:", MAX_LENGTH)

Vocabulary size: 30522
Maximum model length: 512
Training max length we chose: 256


In [72]:
#pick the first training example
sample_text = train_df.iloc[0]["text"]
sample_label = train_df.iloc[0]["label"]

print("Label:", sample_label)
print("\nEmail:")
print(sample_text[:500])

Label: 0

Email:
mumia w wrote on escapenumber escapenumber escapenumber escapenumber escapenumber am mathew snyder wrote in the debugger i've set the 'w' command to watch the variable containing the day being looked for day i would have thought day stays the same throught an iteration of the while loop but the debugger keeps stopping to tell me it switched from 'escapenumber escapenumber escapenumber' for instance to '' i then continue and it tells me it switched back from '' to 'escapenumber escapenumber escap


In [73]:
#then tokenize it
sample_encoding = tokenizer(
    sample_text,
    max_length=MAX_LENGTH,
    truncation=True,
    padding="max_length",
    return_tensors="pt"
)

print("input_ids shape:", sample_encoding["input_ids"].shape)
print("attention_mask shape:", sample_encoding["attention_mask"].shape)

print("\nNumber of real tokens:",
      sample_encoding["attention_mask"].sum().item())

print("Number of padding tokens:",
      MAX_LENGTH - sample_encoding["attention_mask"].sum().item())

input_ids shape: torch.Size([1, 256])
attention_mask shape: torch.Size([1, 256])

Number of real tokens: 254
Number of padding tokens: 2


In [74]:
#in the above code if the email is longer than 256, BERT picks only the 256 and leaves out the other
#Before we commit to MAX_LENGTH=256, though, your question reveals something worth checking: how many emails in our dataset actually exceed 256 tokens?
#If it's 2%, truncating may be perfectly reasonable. If it's 30%, we'd potentially be throwing away a lot of email content.

In [75]:
train_token_lengths = []

for text in train_df["text"]:
    tokens = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False
    )

    train_token_lengths.append(len(tokens["input_ids"]))

print("Number of training emails:", len(train_token_lengths))
print("Shortest email:", min(train_token_lengths))
print("Longest email:", max(train_token_lengths))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (711 > 512). Running this sequence through the model will result in indexing errors


Number of training emails: 66758
Shortest email: 3
Longest email: 48690


In [76]:
import numpy as np

lengths = np.array(train_token_lengths)

print("Median:", np.percentile(lengths, 50))
print("75th percentile:", np.percentile(lengths, 75))
print("90th percentile:", np.percentile(lengths, 90))
print("95th percentile:", np.percentile(lengths, 95))
print("99th percentile:", np.percentile(lengths, 99))

print("\nOver 128:", np.mean(lengths > 128) * 100, "%")
print("Over 256:", np.mean(lengths > 256) * 100, "%")
print("Over 512:", np.mean(lengths > 512) * 100, "%")

Median: 205.0
75th percentile: 423.75
90th percentile: 780.0
95th percentile: 1103.0
99th percentile: 2936.289999999979

Over 128: 67.66080469756433 %
Over 256: 41.17858533808682 %
Over 512: 19.131789448455617 %


Because the dataset contains emails as long as 48,690 tokens, I want to make sure those aren't unusual artifacts, giant concatenated messages, HTML dumps, etc. Let's inspect the longest few examples rather than blindly designing around them.

In [77]:
train_df = train_df.copy()
train_df["token_length"] = train_token_lengths

longest = train_df.sort_values(
    "token_length",
    ascending=False
)[["label", "token_length", "text"]].head(5)

for i, row in longest.iterrows():
    print("=" * 80)
    print("Label:", row["label"])
    print("Token length:", row["token_length"])
    print("\nFirst 1000 characters:")
    print(row["text"][:1000])
    print()

Label: 0
Token length: 48690

First 1000 characters:
enron : a wake - up call
the wall street journal , 12 / 04 / 01
fall of a power giant : dynegy hits back at enron with lawsuit - - - claim is that failed merger entitles concern to get northern natural gas
the wall street journal , 12 / 04 / 01
enron secures $ 1 . 5 billion in new financing - - - j . p . morgan , citigroup pact is first step ; now , lifeline is sought for trading unit
the wall street journal , 12 / 04 / 01
fall of a power giant : burst of chapter 11 filings marks big year for the bankruptcy bar
the wall street journal , 12 / 04 / 01
fall of a power giant : now is the time for creditors to vie against each other
the wall street journal , 12 / 04 / 01
auditor andersen is added to enron 401 ( k ) lawsuit
the wall street journal , 12 / 04 / 01
stadium jinx : what to call enron field ? ` enron folds , ' maybe - - - firms that put their names on arenas hit hard times ; former trans world dome
the wall street journal , 12 /

Next: create a PyTorch Dataset
Rather than tokenizing all 66,758 emails into giant tensors immediately, let's create a Dataset. When PyTorch asks for an email, the dataset will tokenize that email and return its BERT inputs plus its label.

In [78]:
import torch
from torch.utils.data import Dataset

class SpamDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }

In [79]:
#create the three datasets
train_dataset = SpamDataset(
    train_df,
    tokenizer,
    MAX_LENGTH
)

val_dataset = SpamDataset(
    val_df,
    tokenizer,
    MAX_LENGTH
)

test_dataset = SpamDataset(
    test_df,
    tokenizer,
    MAX_LENGTH
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 66758
Validation: 8345
Test: 8345


In [80]:
#inspect one item
sample = train_dataset[0]

print("input_ids:", sample["input_ids"].shape)
print("attention_mask:", sample["attention_mask"].shape)
print("label:", sample["label"])

input_ids: torch.Size([256])
attention_mask: torch.Size([256])
label: tensor(0)


In [81]:
#this tells us that for each dataset item
#we have the input_id, the attention mask and the label


In [82]:
#now we need batches
#During training, processing emails one at a time would be inefficient. Instead, PyTorch's DataLoader groups multiple dataset items together.

from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [83]:
#Notice shuffle=True only for training. We want training examples presented in a shuffled order each epoch.
#There's no need to shuffle validation/test data.

In [84]:
#inspect one batch
batch = next(iter(train_loader))

print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("labels:", batch["label"].shape)

print("\nLabels:")
print(batch["label"])

input_ids: torch.Size([16, 256])
attention_mask: torch.Size([16, 256])
labels: torch.Size([16])

Labels:
tensor([1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1])


In [85]:
#this means we have 16emails, 256 token positions per email
#see the batch level is 16 meaning each label is for an email
#when they enter into the embedding layer we shall have [16, 256, 768]

In [86]:
#create the real classifier
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

print(model)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [87]:
#Near the bottom of the printed architecture (768,2)
#thats the classification head


In [88]:
#send one batch through the classifier
id="p59n2h"
model.eval()

with torch.no_grad():
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["label"]
    )

print("Logits shape:", outputs.logits.shape)
print("Loss:", outputs.loss.item())

print("\nFirst email logits:")
print(outputs.logits[0])

print("\nFirst email probabilities:")
print(torch.softmax(outputs.logits[0], dim=-1))

print("\nTrue label:")
print(batch["label"][0])

Logits shape: torch.Size([16, 2])
Loss: 0.7169296741485596

First email logits:
tensor([0.3503, 0.2233])

First email probabilities:
tensor([0.5317, 0.4683])

True label:
tensor(1)


In [89]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

model = model.to(device)

Device: cuda


In [90]:
#optimizer
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

print(optimizer)

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 2e-05
    maximize: False
    weight_decay: 0.01
)


Perfect. The optimizer is ready. A few things in that output are worth recognizing:
- lr: 2e-05 → our learning rate \(0.00002\)
- weight_decay: 0.01 → AdamW applies weight decay, a regularization mechanism
- betas: (0.9, 0.999) → Adam's running estimates for gradients and squared gradients

In [91]:
#This tells the optimizer that it is allowed to update the trainable parameters throughout our model—including the classifier and BERT itself


In [92]:
#One batch → one parameter update
model.train()

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["label"].to(device)

# 1. Clear old gradients
optimizer.zero_grad()

# 2. Forward pass
outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels
)

loss = outputs.loss

print("Loss before backward:", loss.item())

# 3. Backpropagation
loss.backward()

# 4. Update parameters
optimizer.step()

print("One training step completed.")

Loss before backward: 0.6645911931991577
One training step completed.


Now let's prove that a BERT parameter changed.
We'll inspect one of the parameters we've studied extensively: the query matrix \(W_Q\) in Transformer Layer 1.

In [93]:
# Save Layer 1's query weights BEFORE another training step
WQ_before = model.bert.encoder.layer[0].attention.self.query.weight.detach().clone()

print("WQ shape:", WQ_before.shape)
print("First 5 values before:")
print(WQ_before[0, :5])


WQ shape: torch.Size([768, 768])
First 5 values before:
tensor([-0.0164,  0.0261, -0.0263,  0.0361, -0.0203], device='cuda:0')


In [94]:
# Get another training batch
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["label"].to(device)

# Clear previous gradients
optimizer.zero_grad()

# Forward pass
outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels
)

loss = outputs.loss

# Compute gradients
loss.backward()

# Update parameters
optimizer.step()

print("Loss:", loss.item())

Loss: 0.7135339975357056


In [95]:
#compare

WQ_after = model.bert.encoder.layer[0].attention.self.query.weight.detach().clone()

print("First 5 before:")
print(WQ_before[0, :5])

print("\nFirst 5 after:")
print(WQ_after[0, :5])

print("\nDid WQ change?")
print(not torch.equal(WQ_before, WQ_after))

print("\nMaximum absolute change:")
print(torch.max(torch.abs(WQ_after - WQ_before)).item())

First 5 before:
tensor([-0.0164,  0.0261, -0.0263,  0.0361, -0.0203], device='cuda:0')

First 5 after:
tensor([-0.0164,  0.0261, -0.0263,  0.0361, -0.0203], device='cuda:0')

Did WQ change?
True

Maximum absolute change:
2.008676528930664e-05


In [96]:
#Next we'll build the real training loop, but first reset the model so our experiment starts cleanly.
from transformers import BertForSequenceClassification
from torch.optim import AdamW

# Reload a fresh pretrained BERT classifier
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model = model.to(device)

# Fresh optimizer
optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

print("Fresh model and optimizer ready.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fresh model and optimizer ready.


In [97]:
from tqdm.auto import tqdm

EPOCHS = 1

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}"
    )

    for batch in progress_bar:

        # Move batch to GPU
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        # 1. Clear old gradients
        optimizer.zero_grad()

        # 2. Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        # 3. Backpropagation
        loss.backward()

        # 4. Update weights
        optimizer.step()

        # Track loss
        total_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    average_train_loss = total_loss / len(train_loader)

    print(
        f"\nEpoch {epoch + 1} "
        f"average training loss: {average_train_loss:.4f}"
    )

Epoch 1:   0%|          | 0/4173 [00:00<?, ?it/s]


Epoch 1 average training loss: 0.0522


In [98]:
#evaluate the model on the validation set
#the key difference is model.eval() and with torch.no_grad():

from tqdm.auto import tqdm

model.eval()

total_val_loss = 0
correct = 0
total = 0

with torch.no_grad():

    for batch in tqdm(val_loader, desc="Validation"):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        total_val_loss += loss.item()

        predictions = torch.argmax(logits, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)


average_val_loss = total_val_loss / len(val_loader)
val_accuracy = correct / total

print("Validation loss:", average_val_loss)
print("Validation accuracy:", val_accuracy)
print("Correct:", correct)
print("Total:", total)

Validation:   0%|          | 0/522 [00:00<?, ?it/s]

Validation loss: 0.035633387928895505
Validation accuracy: 0.9900539245056921
Correct: 8262
Total: 8345


In [99]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

all_labels = []
all_predictions = []

model.eval()

with torch.no_grad():

    for batch in tqdm(val_loader, desc="Collecting predictions"):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

In [100]:
cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[3933   21]
 [  62 4329]]


In [101]:
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["Not Spam", "Spam"],
        digits=4
    )
)

              precision    recall  f1-score   support

    Not Spam     0.9845    0.9947    0.9896      3954
        Spam     0.9952    0.9859    0.9905      4391

    accuracy                         0.9901      8345
   macro avg     0.9898    0.9903    0.9900      8345
weighted avg     0.9901    0.9901    0.9901      8345



In [102]:
from sklearn.metrics import confusion_matrix, classification_report

model.eval()

total_test_loss = 0
correct = 0
total = 0

test_labels = []
test_predictions = []

with torch.no_grad():

    for batch in tqdm(test_loader, desc="Testing"):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        predictions = torch.argmax(logits, dim=1)

        # Loss
        total_test_loss += loss.item()

        # Accuracy
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        # Save predictions for detailed metrics
        test_labels.extend(labels.cpu().numpy())
        test_predictions.extend(predictions.cpu().numpy())


average_test_loss = total_test_loss / len(test_loader)
test_accuracy = correct / total

print("Test loss:", average_test_loss)
print("Test accuracy:", test_accuracy)
print("Correct:", correct)
print("Total:", total)

Testing:   0%|          | 0/522 [00:00<?, ?it/s]

Test loss: 0.032876666427310734
Test accuracy: 0.9911324146195326
Correct: 8271
Total: 8345


In [103]:
test_cm = confusion_matrix(
    test_labels,
    test_predictions
)

print("Confusion Matrix:")
print(test_cm)

print("\nClassification Report:")
print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=["Not Spam", "Spam"],
        digits=4
    )
)

Confusion Matrix:
[[3934   20]
 [  54 4337]]

Classification Report:
              precision    recall  f1-score   support

    Not Spam     0.9865    0.9949    0.9907      3954
        Spam     0.9954    0.9877    0.9915      4391

    accuracy                         0.9911      8345
   macro avg     0.9909    0.9913    0.9911      8345
weighted avg     0.9912    0.9911    0.9911      8345



In [104]:
#1. Inspect the spam emails BERT missed
import numpy as np

test_results = test_df.copy().reset_index(drop=True)

test_results["prediction"] = test_predictions

false_negatives = test_results[
    (test_results["label"] == 1) &
    (test_results["prediction"] == 0)
]

print("Number of spam emails missed:", len(false_negatives))

for i, row in false_negatives.head(5).iterrows():
    print("=" * 80)
    print("TRUE LABEL: Spam")
    print("PREDICTION: Not Spam")
    print()
    print(row["text"][:1500])
    print()

Number of spam emails missed: 54
TRUE LABEL: Spam
PREDICTION: Not Spam

june escapenumberth escapenumber tomb raider anniversary w bonus comics escapenumber escapenumber celebrate and commemorate ten years of tomb raider and lara croft with tomb raider anniversary a new action adventure inspired by the original tomb raider video game one of the greatest action adventure games of all time sword of the stars born of blood escapenumber escapenumber defeat a new enemy race in the action packed official add on to sword of the stars tom clancy's ghost recon advanced warfighter escapenumber pre order escapenumber escapenumber building on the foundation of ghost recon advanced warfighter the sequel is a near future warfare masterpiece ghost recon advanced warfighter escapenumber for the pc screams tactics and you'll need to bring your best to bear if you are going to get the ghosts past their latest challenge value zone check out tons of great titles for under escapenumber today caesar iv esca

In [105]:
#test your own email
def predict_email(text):

    model.eval()

    encoding = tokenizer(
        text,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        prediction = torch.argmax(
            probabilities,
            dim=1
        ).item()

    not_spam_probability = probabilities[0][0].item()
    spam_probability = probabilities[0][1].item()

    print("Not Spam probability:", f"{not_spam_probability:.4f}")
    print("Spam probability:", f"{spam_probability:.4f}")

    if prediction == 1:
        print("\nPrediction: SPAM")
    else:
        print("\nPrediction: NOT SPAM")

In [107]:
my_email = """
Hi John,

Just checking whether we're still meeting tomorrow at 10am.
I attached my bank account

Best,
Alice
"""

predict_email(my_email)

Not Spam probability: 0.7923
Spam probability: 0.2077

Prediction: NOT SPAM


In [108]:
my_email = """
URGENT ACTION REQUIRED!

Your bank account has been suspended.
You must verify your account immediately to avoid permanent closure.

Click the link below and enter your banking details,
password and account information to restore access.

Act now. Failure to verify your bank account today
will result in immediate suspension.
"""

predict_email(my_email)

Not Spam probability: 0.0495
Spam probability: 0.9505

Prediction: SPAM


In [109]:
tricky_email = """
Hi John,

I hope you're doing well.

Following up on our conversation from earlier this week,
I've attached the updated document for your review.

When you have a moment, please confirm the account details
listed in the document so we can complete the payment.

There's no immediate rush, but it would be helpful to have
your confirmation before the end of the week.

Thanks again for your help.

Best,
Alice
"""

predict_email(tricky_email)

Not Spam probability: 0.9935
Spam probability: 0.0065

Prediction: NOT SPAM


In [112]:
real_email = """
Hi Aliho,

A degree can tell an employer what you've studied. But what about everything you're capable of doing?

For many careers, practical abilities and the evidence behind them can be just as important as formal qualifications. Employers want to understand what you can contribute, how prepared you are, and whether you have the knowledge to succeed in the role.

That’s why it pays to keep building skills you can actually use and demonstrate. The more you can do, the more you have to show employers beyond your qualifications.

And you don’t need another degree to do it. Start building relevant skills today with our free courses below.
"""

predict_email(real_email)

Not Spam probability: 0.8320
Spam probability: 0.1680

Prediction: NOT SPAM


How to improve the model performance

1. Improve the data. This is often one of the biggest levers.
You might discover mislabeled examples, duplicates, poor-quality examples, insufficient examples of difficult cases, or a mismatch between your training data and the real world.
We actually discovered a potential version of this already. Your model gets ~99% on its test split but classified your real Alison marketing email as not-spam.

So adding representative modern examples—newsletters, phishing messages, promotions, transactional email, personal mail, etc.—could potentially help more than tweaking BERT itself.
This idea is usually called domain adaptation or dealing with distribution/domain shift, depending on exactly what you're doing.

2. Hyperparameter tuning. We chose

lr = 2e-5
batch_size = 16
epochs = 1
max_length = 256

Those weren't laws of nature.
You could systematically compare, for example:

learning rate:  1e-5, 2e-5, 3e-5, 5e-5
epochs:         1, 2, 3
max length:     128, 256, 512

3. Train for more epochs — but watch for overfitting

4. Deal with class imbalance.

5. Adjust the decision threshold.

6. Fix input limitations.

7. Try a different pretrained model.

We used:bert-base-uncased

You could compare against models such as RoBERTa or DistilBERT, or a model pretrained closer to your domain.
But this should usually come after understanding your error patterns. A different architecture won't magically repair bad labels or unrepresentative training data.

8. Regularization and training techniques.